In [20]:
from trials_pipeline import fetch_trials, extract_fields, run_pipeline, build_include_exclude_patterns, classify
import pandas as pd

In [21]:
df1 = pd.read_csv("autoimmune_disease_2020-01-01_to_2023-12-31_trials.csv", keep_default_na=False)
df1["date_window"] = "2020-01-01 to 2023-12-31"

df2 = pd.read_csv("autoimmune_disease_2023-07-01_to_2026-07-01_trials.csv", keep_default_na=False)
df2["date_window"] = "2023-07-01 to 2026-07-01"

df_all = pd.concat([df1, df2], ignore_index=True)
df_all = df_all[df_all["phase"] != "NA"]

In [22]:
lookup = pd.read_csv("industry_sponsor_normalization_USABLE_LOOKUP.csv")
df_all["lead_sponsor_normalized"] = df_all["lead_sponsor"].str.strip().str.lower()
lookup["raw_sponsor_name_normalized"] = lookup["raw_sponsor_name"].str.strip().str.lower()
df_all = df_all.merge(
    lookup[["raw_sponsor_name_normalized", "dashboard_sponsor_name"]],
    left_on="lead_sponsor_normalized",
    right_on="raw_sponsor_name_normalized",
    how="left",
    indicator=True
)
df_all["dashboard_sponsor_name"] = df_all["dashboard_sponsor_name"].fillna(df_all["lead_sponsor"])

In [23]:
from trials_pipeline import build_include_exclude_patterns, classify

mesh_dict = pd.read_csv("mesh_include_exclude_dictionary.csv")
print(mesh_dict["Disease Area"].unique())  # confirm 'autoimmune' shows up before proceeding

mesh_dict = mesh_dict[mesh_dict["Disease Area"] == "autoimmune"]

autoimmune_include_keywords = {
    "RA": ["rheumatoid arthritis", "arthritis, rheumatoid"],
    "MS": ["multiple sclerosis", "relapsing-remitting", "relapsing remitting", "secondary progressive",
           "primary progressive", "relapsing multiple sclerosis", "progressive multiple sclerosis",
           "clinically isolated syndrome"],
    "SLE": ["systemic lupus erythematosus", "lupus erythematosus, systemic", "sle"],
    "Myasthenia gravis": ["myasthenia gravis", "gmg"],
    "TED": ["thyroid eye disease", "graves orbitopathy", "graves ophthalmopathy"],
    "Graves disease": ["graves disease"],
    "Systemic sclerosis": ["systemic sclerosis", "scleroderma", "ssc"],
    "ITP": ["immune thrombocytopenia", "itp"],
    "Sjogren": ["sjogren", "sjögren"],
    "UC": ["ulcerative colitis"],
    "Crohn": ["crohn"],
    "Psoriatic arthritis": ["psoriatic arthritis"],
    "AS": ["ankylosing spondylitis"],
    "Myopathies": ["idiopathic inflammatory myopath", "inflammatory myopathy", "dermatomyositis"],
    "CIDP": ["chronic inflammatory demyelinating poly", "cidp"],
    "NMOSD": ["neuromyelitis optica", "nmosd", "nmo spectrum"],
    "Psoriasis": ["psoriasis"],
    "JIA": ["juvenile idiopathic arthritis"],
    "APS": ["antiphospholipid syndrome", "aps"],
    "Bullous pemphigoid": ["bullous pemphigoid"],
    "AIHA": ["warm autoimmune hemolytic anemia"],
    "Autoimmune encephalitis": ["autoimmune encephalitis"],
    "GBS": ["guillain-barre", "guillain barre"],
    "T1D": ["type 1 diabetes", "diabetes mellitus, type 1", "t1dm", "type i diabetes"],
    "AIH": ["autoimmune hepatitis"],
    "Generic autoimmune": ["autoimmune disease", "autoimmune diseases", "immune system disease"],
    "Vitiligo": ["vitiligo"],
    "Alopecia": ["alopecia areata"],
    "Celiac": ["celiac disease"],
    "Hashimoto": ["hashimoto"],
    "Pemphigus": ["pemphigus"],
}
autoimmune_include_keywords["Additional terms"] = [
    "giant cell arteritis",
    "diabetes mellitus, type 1", "diabetes mellitus type 1", "diabetes type 1", "type1diabetes",
    "type1 diabetes", "diabetes, type i", "diabetes mellitus, type i",
    "graves' disease",
    "arthritis rheumatoid",
    "haemolytic anaemia",
    "igg4", "immunoglobulin g4",
    "still's disease", "still disease", "aosd", "stills disease",
    "cold agglutinin",
    "polymyalgia rheumatica",
    "auto immune",
    "autoimmune hemolytic anemia",
    "b-cell mediated autoimmune",
    "polyneuropathy, inflammatory demyelinating",
    "lupus erythematosus",  # broad catch-all, picks up cutaneous variants and most typos
    "mogad", "myelin oligodendrocyte glycoprotein",
    "spondyloarthritis",
    "chronic spontaneous urticaria",
    "stiff person syndrome", "stiff-person syndrome",
    "thrombocytopenic purpura",
    "autoimmune pulmonary alveolar proteinosis",
    "autoimmune pancreatitis",
    "hidradenitis suppurativa",
    "autoimmune",  # bare catch-all for generic phrasing like "Autoimmune Conditions"
]

include_pattern, exclude_pattern = build_include_exclude_patterns(mesh_dict, autoimmune_include_keywords)

df_all["indication_classification"] = df_all["conditions"].apply(
    lambda c: classify(c, include_pattern, exclude_pattern, match_label="true_autoimmune")
)
df_all["is_drug_biologic"] = df_all["intervention_types"].str.contains("DRUG|BIOLOGICAL", na=False)

<StringArray>
['cardiovascular', 'kidney', 'diabetes', 'liver', 'obesity', 'autoimmune']
Length: 6, dtype: str


In [24]:
review_list = (
    df_all[df_all["indication_classification"] == "needs_review"]["conditions"]
    .str.split("|").explode().value_counts()
)
review_list.to_csv("autoimmune_conditions_needing_review.csv")

In [25]:
# 1. Phase filter
df_phased = df_all[~df_all["phase"].isin(["BLANK", "PHASE2,PHASE3"])]

# 2. Observational split + true-autoimmune-drug filter
df_observational = df_all[
    (df_all["study_type"] == "OBSERVATIONAL")
    & (df_all["indication_classification"] == "true_autoimmune")
]
df_true_autoimmune_drug = df_phased[
    (df_phased["indication_classification"] == "true_autoimmune")
    & (df_phased["is_drug_biologic"])
]

# 3. Company and indication views
company_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2"]
company_late_phases = ["PHASE3", "PHASE4"]
indication_early_phases = ["EARLY_PHASE1", "PHASE1", "PHASE1,PHASE2", "PHASE2"]
indication_late_phases = ["PHASE3", "PHASE4"]

df_company_early = df_true_autoimmune_drug[df_true_autoimmune_drug["phase"].isin(company_early_phases)]
df_company_late = df_true_autoimmune_drug[df_true_autoimmune_drug["phase"].isin(company_late_phases)]

top20_companies_all_phases = (
    df_true_autoimmune_drug.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_companies_early_phase = (
    df_company_early.groupby("dashboard_sponsor_name")["nct_id"]
    .nunique().sort_values(ascending=False).head(20)
)
top20_by_window = (
    df_true_autoimmune_drug.groupby(["date_window", "dashboard_sponsor_name"])["nct_id"]
    .nunique().reset_index()
    .sort_values(["date_window", "nct_id"], ascending=[True, False])
    .groupby("date_window").head(20)
)

df_indications = df_true_autoimmune_drug.assign(
    conditions=df_true_autoimmune_drug["conditions"].str.split("|")
).explode("conditions")

top_indications_early = (
    df_indications[df_indications["phase"].isin(indication_early_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)
top_indications_late = (
    df_indications[df_indications["phase"].isin(indication_late_phases)]
    .groupby("conditions")["nct_id"].nunique()
    .sort_values(ascending=False).head(20)
)

# 4. Exports
top20_companies_all_phases.to_csv("autoimmune_top20_companies_all_phases.csv")
top20_companies_early_phase.to_csv("autoimmune_top20_companies_early_phase.csv")
top20_by_window.to_csv("autoimmune_top20_companies_by_date_window.csv", index=False)
top_indications_early.to_csv("autoimmune_top_indications_early_phase.csv")
top_indications_late.to_csv("autoimmune_top_indications_late_phase.csv")
df_observational.to_csv("autoimmune_observational_studies.csv", index=False)

# 5. Tableau prep
df_all.to_csv("autoimmune_full_dataset_for_tableau.csv", index=False)

# 6. Collaborators explode
df_collab = df_all.assign(
    collaborators=df_all["collaborators"].str.split("|")
).explode("collaborators")
df_collab = df_collab[df_collab["collaborators"].notna() & (df_collab["collaborators"] != "")]
df_collab.to_csv("autoimmune_collaborations.csv", index=False)